In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic-prediction-scc-study-group/titanic_train.csv
/kaggle/input/competitions/titanic-prediction-scc-study-group/sample submission.csv
/kaggle/input/competitions/titanic-prediction-scc-study-group/titanic_test.csv


# EDA（Exploratory Data Analysis）
探索的データ分析

データの参照

In [2]:
import pandas as pd

train_csv = pd.read_csv("/kaggle/input/competitions/titanic-prediction-scc-study-group/titanic_train.csv")
test_csv = pd.read_csv("/kaggle/input/competitions/titanic-prediction-scc-study-group/sample submission.csv")

train_csv.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived
0,693,3,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,NaN,S,1
1,482,2,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,NaN,S,0
2,528,1,"Farthing, Mr. John",male,NaN,0,0,PC 17483,221.7792,C95,S,0
3,856,3,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,NaN,S,1
4,802,2,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,NaN,S,1


# 目的変数”Survived"の割合

In [3]:
Surv_sum = train_csv.Survived.sum()
Surv_sum_ratio = Surv_sum / len(train_csv)
print(f"{Surv_sum_ratio:.3}")

0.383


実際に生存した人間の数は、およそ38%

# それぞれの特徴量の欠損数

In [4]:
train_csv.info()
print()
print(train_csv[["Age","Cabin"]].isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 712 entries, 0 to 711
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  712 non-null    int64  
 1   Pclass       712 non-null    int64  
 2   Name         712 non-null    object 
 3   Sex          712 non-null    object 
 4   Age          575 non-null    float64
 5   SibSp        712 non-null    int64  
 6   Parch        712 non-null    int64  
 7   Ticket       712 non-null    object 
 8   Fare         712 non-null    float64
 9   Cabin        160 non-null    object 
 10  Embarked     710 non-null    object 
 11  Survived     712 non-null    int64  
dtypes: float64(2), int64(5), object(5)
memory usage: 66.9+ KB

Age      137
Cabin    552
dtype: int64


"Age","Caabin"の2つの特徴量が、欠損が多いことがわかる。
それぞれ137個と、552個の欠損あり。

・float,int型　→　7つ

・object型　→　5つ

# 低カーディナリティ特徴量

ここで、低カーディナリティであれば、そのuniqueごとの目的変数に対する相関が比較的容易に見られる可能性があるので、
低カーディナリティ列における生存率を調べて、どの程度の相関があるかを見ていく。

In [5]:
train_csv.nunique()

PassengerId    712
Pclass           3
Name           712
Sex              2
Age             85
SibSp            7
Parch            7
Ticket         571
Fare           226
Cabin          127
Embarked         3
Survived         2
dtype: int64

低カーディナリティに該当する、”Pclass(チケットのクラス/客室階級)"、”Sex(性別）"、”Embarked(出港地)の3つについて、それぞれ見ていく。

それぞれの特徴量でunique値ごとにグループ分けを行い、そのuniqueごとの生存率を見てみる。

In [6]:
low_car = ["Pclass","Sex","Embarked"]

for x in low_car:
    low_car_ratio = train_csv.groupby(x)["Survived"].mean()*100
    print("各ユニーク値における生存率")
    print(low_car_ratio.apply(lambda v : f"{v:.2f}%"))
    print()


各ユニーク値における生存率
Pclass
1    64.91%
2    44.67%
3    24.30%
Name: Survived, dtype: object

各ユニーク値における生存率
Sex
female    74.31%
male      18.52%
Name: Survived, dtype: object

各ユニーク値における生存率
Embarked
C    56.12%
Q    43.64%
S    32.75%
Name: Survived, dtype: object



>"Pclass"においては、1のuniqueの生存率が3に対して実に3倍
>”Sex"においては、femaleのuniqueの生存率がmaleの実に4倍
>”Embarked"においては、Cのuniqueの生存率がSの約1.8倍

全体の生存率は先ほどの通り、38%だったのに対し、"Pclass"の「1」に該当するものでは生存率が全体に比べて1.8倍ほど高く、また、”Sex"の「Female」に該当するものは全体に比べて、約2倍ほど生存率が高くなっている。

この2つの突出している”Pclass"と”Sex"は、目的変数に対して強い相関を持っている可能性が高い。

# 数値列